# TurkMotifTesettür — SDXL LoRA for Turkish Motif Modest Fashion

This notebook documents the full pipeline of a graduation project that trains a
domain-specific **LoRA (Low-Rank Adaptation)** adapter on **Stable Diffusion XL (SDXL)**
to generate modern modest-fashion designs inspired by traditional Turkish motifs
(Ottoman tulip, Iznik tile, rumi, and saz-leaf patterns).

The trained adapter is activated through the trigger token **`turkmotiftesettur`**.

**Pipeline stages:**
1. Environment setup (dependencies + Kohya sd-scripts)
2. Base model download
3. Dataset preparation (Kohya repeat format + caption enrichment)
4. Project folder structure
5. Sample prompts for training-time previews
6. Training experiments (Debug → v1 → Exp01 → Exp02)
7. Inference (image generation)
8. Quantitative evaluation (CLIP, DINOv2, FID/KID, LPIPS)

> **Note:** Paths point to Google Drive (`/content/drive/MyDrive/bitirme_lora`).
> Large files (base model, datasets, trained weights) are **not** included in the
> repository; only the source code is tracked.


## 1. Environment Setup

Mount Google Drive, check the GPU, and install all required libraries together with
the Kohya `sd-scripts` training framework.


**1.1. Mount Google Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**1.2. GPU check, dependency installation, and Kohya sd-scripts**

In [ ]:
#GPU kontrolü , kohya ayarlama colab a indime
!nvidia-smi
!pip install -q torch torchvision xformers --index-url https://download.pytorch.org/whl/cu121
!pip install -q accelerate transformers diffusers[torch] ftfy opencv-python bitsandbytes safetensors
!git clone https://github.com/kohya-ss/sd-scripts.git
%cd /content/sd-scripts
!pip install -q -r requirements.txt
!pip install -q .

## 2. Base Model Download

The SDXL base 1.0 model is downloaded from the Hugging Face Hub. Two options are shown:
downloading to the fast local Colab disk, or to Google Drive for persistent storage.
A Hugging Face login is optional (only needed for higher rate limits).


**2.1. (Optional) Hugging Face login**

In [ ]:
from huggingface_hub import login
login()

**2.2. Download base model to local Colab disk (fast, temporary)**

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path
#modeli hugging den colaba çekiyor.
MODEL_DIR = Path("/content/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

base_model_path = hf_hub_download(
    repo_id="stabilityai/stable-diffusion-xl-base-1.0",
    filename="sd_xl_base_1.0.safetensors",
    local_dir=MODEL_DIR
)

print("Base model yolu:", base_model_path)

**2.3. Download base model to Google Drive (persistent)**

In [ ]:
from pathlib import Path
from huggingface_hub import hf_hub_download, notebook_login
#modeli drive a çekiyor.
BASE_DIR = Path("/content/drive/MyDrive/bitirme_lora")
MODEL_DIR = BASE_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Eğer token isterse bu çalışır.
# notebook_login()

base_model_path = hf_hub_download(
    repo_id="stabilityai/stable-diffusion-xl-base-1.0",
    filename="sd_xl_base_1.0.safetensors",
    local_dir=MODEL_DIR
)

print("Base model yolu:", base_model_path)

## 3. Project Folder Structure

Define the directory layout used throughout the project. The final experiment (Exp02)
is organized under a single `experiments/02_second_training_motif_stronger/` folder
containing the dataset, weights, logs, and samples.


**3.1. Initial folder layout (v1)**

In [ ]:
from pathlib import Path
#mimari yapının klasörlerini ayarlıyor
BASE_DIR = Path("/content/drive/MyDrive/bitirme_lora")


TRAIN_DIR = BASE_DIR / "run_dataset"
OUTPUT_DIR = BASE_DIR / "output" / "turkmotiftesettur_sdxl_lora_v1"
LOG_DIR = BASE_DIR / "logs" / "turkmotiftesettur_sdxl_lora_v1"
SAMPLE_PROMPTS_FILE = BASE_DIR / "sample_prompts.txt"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print("Train dir:", TRAIN_DIR)
print("Output dir:", OUTPUT_DIR)
print("Log dir:", LOG_DIR)

'''  bitirme_lora/
├── models/
│   └── sd_xl_base_1.0.safetensors
├── run_dataset/
│   ├── 10_turkmotiftesettur_clothes/
│   ├── 8_turkmotiftesettur_plain_forms/
│   └── 4_turkmotiftesettur_motifs/
├── output/
│   └── turkmotiftesettur_sdxl_lora_v1/
├── logs/
│   └── turkmotiftesettur_sdxl_lora_v1/
└── sample_prompts.txt                  '''

**3.2. Experiment-based folder layout (Exp02)**

In [ ]:
from pathlib import Path
#klasör yapısını güncelleme

BASE_DIR = Path("/content/drive/MyDrive/bitirme_lora")

EXP2_DIR = BASE_DIR / "experiments" / "02_second_training_motif_stronger"

DATASET_DIR = EXP2_DIR / "dataset"
OUTPUT_DIR = EXP2_DIR / "weights"
LOG_DIR = EXP2_DIR / "logs"
SAMPLE_DIR = EXP2_DIR / "samples"

DATASET_CONFIG = EXP2_DIR / "dataset_config_exp02.toml"
SAMPLE_PROMPTS_FILE = EXP2_DIR / "sample_prompts.txt"
CONSOLE_LOG = LOG_DIR / "console_train_log_dataset_config.txt"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset:", DATASET_DIR)
print("Dataset var mı:", DATASET_DIR.exists())
print("Output:", OUTPUT_DIR)
print("Log:", LOG_DIR)

'''
bitirme_lora/
├── models/
│   └── sd_xl_base_1.0.safetensors
├── run_dataset/                          ← v1'den kalan
├── experiments/
│   └── 02_second_training_motif_stronger/
│       ├── dataset/                      ← eğitim görselleri
│       ├── weights/                      ← çıkacak LoRA dosyaları
│       ├── logs/
│       │   └── console_train_log_dataset_config.txt
│       ├── samples/                      ← eğitim sırasında üretilen örnekler
│       ├── dataset_config_exp02.toml     ← Kohya config
│       └── sample_prompts.txt            ← örnek üretim promptları
'''

## 4. Dataset Preparation

The dataset consists of image–caption pairs in three categories: motif-bearing
garments, plain modest forms, and pure motif references. They are organized into the
Kohya *repeat* folder format, where the number prefixing each folder name sets how many
times its images are repeated during training.

**First dataset (v1):** simple copy of pairs.
**Second dataset (Exp02):** increased motif count (40 → 80), category-specific
**caption enrichment**, and a safety-guarded folder cleanup.


**4.1. First dataset build (v1) — simple pair copy**

In [ ]:
import os
import random
import shutil
from pathlib import Path

# =========================
# AYARLAR
# =========================

BASE_DIR = Path("/content/drive/MyDrive/bitirme_lora")

CLOTHES_DATASET_DIR = BASE_DIR / "turkmotiftesettur_full_dataset_updated_v2"
MOTIF_DATASET_DIR = BASE_DIR / "BitirmeLora_checked_fixed"

RUN_DATASET_DIR = BASE_DIR / "run_dataset"

# Eğitimde kaç motif kullanılacağını belirleme
MOTIF_COUNT = 40

# Rastgele seçim sabit kalsın diye
SEED = 42

IMAGE_EXTENSIONS = [".png", ".jpg", ".jpeg", ".webp"]

# =========================
# YARDIMCI FONKSİYONLAR
# =========================
# Bir klasörün içindeki her görsele bakıp, yanında aynı isimli .txt dosyası var mı diye kontrol ediyor.
def find_image_caption_pairs(folder):
    folder = Path(folder)
    pairs = []

    for file in folder.rglob("*"):
        if file.suffix.lower() in IMAGE_EXTENSIONS:
            txt_file = file.with_suffix(".txt")
            if txt_file.exists():
                pairs.append((file, txt_file))
            else:
                print(f"Caption eksik: {file}")

    return pairs

#Eşleşmiş çiftleri hedef klasöre kopyalarken hepsini tutarlı biçimde yeniden adlandırıyor.
def copy_pairs(pairs, target_folder, prefix):
    target_folder = Path(target_folder)
    target_folder.mkdir(parents=True, exist_ok=True)

    for idx, (img_path, txt_path) in enumerate(pairs, start=1):
        new_name = f"{prefix}_{idx:04d}"
        new_img = target_folder / f"{new_name}{img_path.suffix.lower()}"
        new_txt = target_folder / f"{new_name}.txt"

        shutil.copy2(img_path, new_img)
        shutil.copy2(txt_path, new_txt)


def clear_folder(folder):
    folder = Path(folder)
    if folder.exists():
        shutil.rmtree(folder)
    folder.mkdir(parents=True, exist_ok=True)


# =========================
# KLASÖR YOLLARINI BUL
# =========================

clothes_root = CLOTHES_DATASET_DIR

# Bazı ziplerde iç içe klasör oluşabiliyor. Bu yüzden olası yolu kontrol ediyor.
possible_train_dirs = [
    clothes_root / "train",
    clothes_root / "turkmotiftesettur_full_dataset" / "train",
    clothes_root / "turkmotiftesettur_full_dataset_v2" / "train",
]

train_dir = None
for p in possible_train_dirs:
    if p.exists():
        train_dir = p
        break

if train_dir is None:
    raise FileNotFoundError(
        "Kıyafet train klasörü bulunamadı. Drive'daki klasör yapısını kontrol et."
    )

motifli_dir = train_dir / "01_motifli_ve_geleneksel_kiyafetler"
sade_dir = train_dir / "02_sade_modern_tesettur_formlari"

if not motifli_dir.exists():
    raise FileNotFoundError(f"Motifli kıyafet klasörü bulunamadı: {motifli_dir}")

if not sade_dir.exists():
    raise FileNotFoundError(f"Sade modern form klasörü bulunamadı: {sade_dir}")

motif_root = MOTIF_DATASET_DIR

# Motifler bazen direkt klasörde, bazen iç klasörde olabilir.
possible_motif_dirs = [
    motif_root,
    motif_root / "BitirmeLora_checked_fixed",
    motif_root / "BitirmeLora",
]

motif_dir = None
for p in possible_motif_dirs:
    if p.exists() and len(find_image_caption_pairs(p)) > 0:
        motif_dir = p
        break

if motif_dir is None:
    raise FileNotFoundError(
        "Motif klasörü bulunamadı veya içinde eşleşen görsel-caption yok."
    )

print("Bulunan klasörler:")
print("Motifli kıyafet:", motifli_dir)
print("Sade modern form:", sade_dir)
print("Motif referans:", motif_dir)

# =========================
# GÖRSEL-CAPTION EŞLEŞMELERİNİ OKU
# =========================

motifli_pairs = find_image_caption_pairs(motifli_dir)
sade_pairs = find_image_caption_pairs(sade_dir)
motif_pairs = find_image_caption_pairs(motif_dir)

print("\nBulunan eşleşmeler:")
print("Motifli / geleneksel kıyafet:", len(motifli_pairs))
print("Sade modern tesettür formu:", len(sade_pairs))
print("Motif referansı:", len(motif_pairs))

if len(motif_pairs) < MOTIF_COUNT:
    print(f"Uyarı: {MOTIF_COUNT} motif istedin ama sadece {len(motif_pairs)} motif var.")
    MOTIF_COUNT = len(motif_pairs)

# =========================
# MOTİF SEÇİMİ
# =========================

random.seed(SEED)
selected_motifs = random.sample(motif_pairs, MOTIF_COUNT)

# =========================
# RUN DATASET OLUŞTUR
# =========================

clear_folder(RUN_DATASET_DIR)

# Kohya repeat klasör mantığı:
# 10 = ana kıyafetler daha baskın
# 8 = sade formlar destekleyici
# 4 = motifler destekleyici
target_motifli = RUN_DATASET_DIR / "10_turkmotiftesettur_clothes"
target_sade = RUN_DATASET_DIR / "8_turkmotiftesettur_plain_forms"
target_motif = RUN_DATASET_DIR / "4_turkmotiftesettur_motifs"

copy_pairs(motifli_pairs, target_motifli, "clothes")
copy_pairs(sade_pairs, target_sade, "plain")
copy_pairs(selected_motifs, target_motif, "motif")

print("\nRun dataset oluşturuldu:")
print(RUN_DATASET_DIR)

print("\nKopyalanan görsel sayıları:")
print("10_turkmotiftesettur_clothes:", len(motifli_pairs))
print("8_turkmotiftesettur_plain_forms:", len(sade_pairs))
print("4_turkmotiftesettur_motifs:", len(selected_motifs))

# =========================
# SON KONTROL
# =========================

total_images = 0
total_txt = 0

for folder in RUN_DATASET_DIR.iterdir():
    if folder.is_dir():
        images = [f for f in folder.iterdir() if f.suffix.lower() in IMAGE_EXTENSIONS]
        txts = [f for f in folder.iterdir() if f.suffix.lower() == ".txt"]

        total_images += len(images)
        total_txt += len(txts)

        print(f"\n{folder.name}")
        print("Görsel:", len(images))
        print("Caption:", len(txts))

print("\nTOPLAM")
print("Görsel:", total_images)
print("Caption:", total_txt)

if total_images == total_txt:
    print("\n✅ Dataset eğitim için hazır.")
else:
    print("\n❌ Görsel-caption sayısı eşleşmiyor. Kontrol gerekli.")

**4.2. Validate dataset (compare image vs caption counts)**

In [ ]:
IMAGE_EXTS = [".jpg", ".jpeg", ".png", ".webp"]
#görsel ve caption sayılarını karşılaştırıyo.
for folder in DATASET_DIR.iterdir():
    if folder.is_dir():
        images = [f for f in folder.iterdir() if f.suffix.lower() in IMAGE_EXTS]
        txts = [f for f in folder.iterdir() if f.suffix.lower() == ".txt"]
        print(folder.name, "Görsel:", len(images), "Caption:", len(txts))

**4.3. Second dataset build (Exp02) — caption enrichment**

Each caption receives a category-specific prefix (e.g. *ottoman tulip motif, iznik tile
inspired embroidery, rumi motif, saz leaf motif*) so the model learns to associate these
specific terms with the visual motifs. A `safe_clear_folder` guard prevents accidental
deletion of critical directories.


In [ ]:
#ikinci eğitim için ikinci dataset
import os
import random
import shutil
from pathlib import Path

# =========================
# AYARLAR
# =========================

BASE_DIR = Path("/content/drive/MyDrive/bitirme_lora")

CLOTHES_DATASET_DIR = BASE_DIR / "turkmotiftesettur_full_dataset_updated_v2"
MOTIF_DATASET_DIR = BASE_DIR / "BitirmeLora_checked_fixed"

EXP2_DIR = BASE_DIR / "experiments" / "02_second_training_motif_stronger"
RUN_DATASET_DIR = EXP2_DIR / "dataset"

# İlk eğitimde 40 motif vardı. Motifler zayıf kaldığı için artırıyoruz.
MOTIF_COUNT = 80

SEED = 42

IMAGE_EXTENSIONS = [".png", ".jpg", ".jpeg", ".webp"]

# =========================
# GÜVENLİ TEMİZLEME
# =========================

def safe_clear_folder(folder):
    folder = Path(folder)

    # Güvenlik: yanlışlıkla output veya ana klasör silinmesin
    forbidden_names = ["output", "logs", "experiments", "bitirme_lora", "MyDrive"]
    if folder.name in forbidden_names:
        raise ValueError(f"Bu klasör güvenlik nedeniyle silinmez: {folder}")

    if "02_second_training_motif_stronger/dataset" not in str(folder):
        raise ValueError(f"Beklenmeyen dataset yolu, silme durduruldu: {folder}")

    if folder.exists():
        shutil.rmtree(folder)

    folder.mkdir(parents=True, exist_ok=True)

# =========================
# YARDIMCI FONKSİYONLAR
# =========================

def find_image_caption_pairs(folder):
    folder = Path(folder)
    pairs = []

    for file in folder.rglob("*"):
        if file.suffix.lower() in IMAGE_EXTENSIONS:
            txt_file = file.with_suffix(".txt")
            if txt_file.exists():
                pairs.append((file, txt_file))
            else:
                print(f"Caption eksik: {file}")

    return pairs


def enhance_caption(original_caption, group_type):
    text = original_caption.strip()
    text = text.replace("\n", " ")
    text = text.replace(",,", ",")

    trigger = "turkmotiftesettur"

    if group_type == "motifli_clothes":
        prefix = (
            "turkmotiftesettur, modern modest fashion, hijab, long sleeve, "
            "full length dress, traditional turkish textile patterns, "
            "ottoman tulip motif, iznik tile inspired embroidery, "
            "rumi motif, saz leaf motif, anatolian embroidery"
        )

    elif group_type == "plain_forms":
        prefix = (
            "turkmotiftesettur, modern modest fashion, hijab, long sleeve, "
            "full length dress, clean modest silhouette, simple tesettur outfit"
        )

    elif group_type == "motif_reference":
        prefix = (
            "turkmotiftesettur, turkish ornamental motif reference, "
            "traditional turkish textile pattern, ottoman tulip motif, "
            "iznik tile pattern, rumi motif, saz leaf motif, anatolian motif"
        )

    else:
        prefix = trigger

    if not text.lower().startswith(trigger):
        text = prefix + ", " + text
    else:
        text = text

    return text


def copy_pairs_with_enhanced_caption(pairs, target_folder, prefix, group_type):
    target_folder = Path(target_folder)
    target_folder.mkdir(parents=True, exist_ok=True)

    for idx, (img_path, txt_path) in enumerate(pairs, start=1):
        new_name = f"{prefix}_{idx:04d}"
        new_img = target_folder / f"{new_name}{img_path.suffix.lower()}"
        new_txt = target_folder / f"{new_name}.txt"

        shutil.copy2(img_path, new_img)

        original_caption = txt_path.read_text(encoding="utf-8", errors="ignore")
        new_caption = enhance_caption(original_caption, group_type)

        new_txt.write_text(new_caption, encoding="utf-8")


# =========================
# KLASÖR YOLLARINI BUL
# =========================

clothes_root = CLOTHES_DATASET_DIR

possible_train_dirs = [
    clothes_root / "train",
    clothes_root / "turkmotiftesettur_full_dataset" / "train",
    clothes_root / "turkmotiftesettur_full_dataset_v2" / "train",
]

train_dir = None
for p in possible_train_dirs:
    if p.exists():
        train_dir = p
        break

if train_dir is None:
    raise FileNotFoundError("Kıyafet train klasörü bulunamadı.")

motifli_dir = train_dir / "01_motifli_ve_geleneksel_kiyafetler"
sade_dir = train_dir / "02_sade_modern_tesettur_formlari"

if not motifli_dir.exists():
    raise FileNotFoundError(f"Motifli kıyafet klasörü bulunamadı: {motifli_dir}")

if not sade_dir.exists():
    raise FileNotFoundError(f"Sade modern form klasörü bulunamadı: {sade_dir}")

motif_root = MOTIF_DATASET_DIR

possible_motif_dirs = [
    motif_root,
    motif_root / "BitirmeLora_checked_fixed",
    motif_root / "BitirmeLora",
]

motif_dir = None
for p in possible_motif_dirs:
    if p.exists() and len(find_image_caption_pairs(p)) > 0:
        motif_dir = p
        break

if motif_dir is None:
    raise FileNotFoundError("Motif klasörü bulunamadı veya caption eşleşmesi yok.")

print("Bulunan klasörler:")
print("Motifli kıyafet:", motifli_dir)
print("Sade modern form:", sade_dir)
print("Motif referans:", motif_dir)

# =========================
# EŞLEŞMELER
# =========================

motifli_pairs = find_image_caption_pairs(motifli_dir)
sade_pairs = find_image_caption_pairs(sade_dir)
motif_pairs = find_image_caption_pairs(motif_dir)

print("\nBulunan eşleşmeler:")
print("Motifli / geleneksel kıyafet:", len(motifli_pairs))
print("Sade modern tesettür formu:", len(sade_pairs))
print("Motif referansı:", len(motif_pairs))

if len(motif_pairs) < MOTIF_COUNT:
    print(f"Uyarı: {MOTIF_COUNT} motif istedin ama sadece {len(motif_pairs)} motif var.")
    MOTIF_COUNT = len(motif_pairs)

random.seed(SEED)
selected_motifs = random.sample(motif_pairs, MOTIF_COUNT)

# =========================
# RUN DATASET OLUŞTUR
# =========================

safe_clear_folder(RUN_DATASET_DIR)

# İkinci eğitim için daha motif odaklı repeat yapısı
target_motifli = RUN_DATASET_DIR / "12_turkmotiftesettur_motif_clothes"
target_sade = RUN_DATASET_DIR / "4_turkmotiftesettur_plain_forms"
target_motif = RUN_DATASET_DIR / "8_turkmotiftesettur_motif_references"

copy_pairs_with_enhanced_caption(
    motifli_pairs,
    target_motifli,
    "clothes",
    "motifli_clothes"
)

copy_pairs_with_enhanced_caption(
    sade_pairs,
    target_sade,
    "plain",
    "plain_forms"
)

copy_pairs_with_enhanced_caption(
    selected_motifs,
    target_motif,
    "motif",
    "motif_reference"
)

print("\nİkinci eğitim dataset oluşturuldu:")
print(RUN_DATASET_DIR)

# =========================
# SON KONTROL
# =========================

total_images = 0
total_txt = 0

for folder in RUN_DATASET_DIR.iterdir():
    if folder.is_dir():
        images = [f for f in folder.iterdir() if f.suffix.lower() in IMAGE_EXTENSIONS]
        txts = [f for f in folder.iterdir() if f.suffix.lower() == ".txt"]

        total_images += len(images)
        total_txt += len(txts)

        print(f"\n{folder.name}")
        print("Görsel:", len(images))
        print("Caption:", len(txts))

print("\nTOPLAM")
print("Görsel:", total_images)
print("Caption:", total_txt)

if total_images == total_txt:
    print("\n✅ İkinci eğitim dataset hazır.")
else:
    print("\n❌ Görsel-caption sayısı eşleşmiyor.")

## 5. Kohya TOML Dataset Configuration

Instead of passing dataset settings on the command line, Exp02 uses a TOML configuration
file. This allows per-subset control of `num_repeats`, `class_tokens`, and resolution.


**5.1. Write the TOML dataset config**

In [ ]:
#kohyanın desteklediği toml dosya formatı ile dataseti hazırlama
motif_clothes_dir = DATASET_DIR / "12_turkmotiftesettur_motif_clothes"
plain_forms_dir = DATASET_DIR / "4_turkmotiftesettur_plain_forms"
motif_refs_dir = DATASET_DIR / "8_turkmotiftesettur_motif_references"

for p in [motif_clothes_dir, plain_forms_dir, motif_refs_dir]:
    print(p, "var mı:", p.exists())

toml_text = f"""
[general]
shuffle_caption = true
caption_extension = ".txt"
keep_tokens = 1

[[datasets]]
resolution = [1024, 1024]
batch_size = 2
enable_bucket = true
bucket_no_upscale = true
bucket_reso_steps = 64

[[datasets.subsets]]
image_dir = "{motif_clothes_dir}"
num_repeats = 12
class_tokens = "turkmotiftesettur modern modest fashion hijab turkish motif clothes"
caption_extension = ".txt"

[[datasets.subsets]]
image_dir = "{plain_forms_dir}"
num_repeats = 4
class_tokens = "turkmotiftesettur modern modest fashion hijab plain modest silhouette"
caption_extension = ".txt"

[[datasets.subsets]]
image_dir = "{motif_refs_dir}"
num_repeats = 8
class_tokens = "turkmotiftesettur turkish ornamental motif reference iznik ottoman anatolian pattern"
caption_extension = ".txt"
""".strip()

DATASET_CONFIG.write_text(toml_text, encoding="utf-8")

print("Dataset config yazıldı:")
print(DATASET_CONFIG)
print("\nİçerik:")
print(DATASET_CONFIG.read_text(encoding="utf-8"))

**5.2. Verify config file and count dataset images**

In [ ]:
#drive da toml dosyası ve resimlerin kontrolü için
print("Dataset config:", DATASET_CONFIG)
print("Config var mı:", DATASET_CONFIG.exists())

!find "{DATASET_DIR}" -maxdepth 2 -type f \( -iname "*.jpg" -o -iname "*.jpeg" -o -iname "*.png" -o -iname "*.webp" \) | wc -l

## 6. Sample Prompts for Training Previews

During training, Kohya periodically generates preview images from these prompts so the
model's progress can be monitored. The prompts include generation parameters
(`--w`, `--h`, `--l`, `--s`).


In [ ]:
sample_prompts = """
turkmotiftesettur, elegant modern modest fashion, long modest dress, ottoman tulip motifs, iznik tile inspired embroidery, rumi and saz leaf patterns, blue and gold textile details, full body, fashion editorial --w 1024 --h 1024 --l 7 --s 30
turkmotiftesettur, modest evening dress, iznik tile inspired floral embroidery, white and blue patterns, hijab, full body, luxury fashion photography --w 1024 --h 1024 --l 7 --s 30
turkmotiftesettur, modern modest outfit, anatolian geometric motifs, embroidered long tunic and skirt, hijab, full body, studio fashion photo --w 1024 --h 1024 --l 7 --s 30
turkmotiftesettur, kaftan style modest gown, ottoman inspired gold embroidery, elegant hijab, full body, luxury fashion lookbook --w 1024 --h 1024 --l 7 --s 30
""".strip()

SAMPLE_PROMPTS_FILE.write_text(sample_prompts, encoding="utf-8")

print("Sample prompt hazır:", SAMPLE_PROMPTS_FILE)

## 7. Model Training

Training evolved through four stages, each informed by the previous one:

| Stage | network_dim | resolution | text encoder | duration | result |
|-------|-------------|------------|--------------|----------|--------|
| Debug | 8  | 768  | on  | 1 epoch  | did not learn (capacity too low) |
| v1    | 16 | 1024 | on  | 10 epoch | CUDA out of memory (T4) |
| Exp01 | 16 | 768  | off | 4 epoch  | weak motif identity (generic embroidery) |
| **Exp02** | **32** | **1024** | **on** | **1800 steps** | **success** |

The cells below preserve each stage for reproducibility and to document the reasoning
behind the final configuration.


**7.1. Path setup for first training runs**

In [ ]:
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/bitirme_lora")

TRAIN_DIR = BASE_DIR / "run_dataset"
OUTPUT_DIR = BASE_DIR / "output" / "turkmotiftesettur_sdxl_lora_v1"
LOG_DIR = BASE_DIR / "logs" / "turkmotiftesettur_sdxl_lora_v1"
SAMPLE_PROMPTS_FILE = BASE_DIR / "sample_prompts.txt"
CONSOLE_LOG = LOG_DIR / "console_train_log.txt"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print("Train:", TRAIN_DIR)
print("Output:", OUTPUT_DIR)
print("Log:", CONSOLE_LOG)

**7.2. Debug run** — minimal config (`network_dim=8`, 1 epoch) used only to verify the
training pipeline runs. The model did not learn meaningful motif identity.

In [ ]:
#modelin öğrenmediği lora

import os

os.chdir("/content/sd-scripts")

!python sdxl_train_network.py \
  --pretrained_model_name_or_path="{base_model_path}" \
  --train_data_dir="{TRAIN_DIR}" \
  --output_dir="{OUTPUT_DIR}" \
  --logging_dir="{LOG_DIR}" \
  --output_name="turkmotiftesettur_sdxl_lora_debug" \
  --save_model_as="safetensors" \
  --network_module=networks.lora \
  --network_dim=8 \
  --network_alpha=8 \
  --resolution=768,768 \
  --enable_bucket \
  --bucket_no_upscale \
  --min_bucket_reso=256 \
  --max_bucket_reso=1024 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --learning_rate=1e-4 \
  --unet_lr=1e-4 \
  --text_encoder_lr=5e-6 \
  --lr_scheduler=cosine \
  --optimizer_type=AdamW8bit \
  --max_train_epochs=1 \
  --save_every_n_epochs=1 \
  --mixed_precision=fp16 \
  --save_precision=fp16 \
  --xformers \
  --gradient_checkpointing \
  --cache_latents \
  --cache_latents_to_disk \
  --caption_extension=.txt \
  --shuffle_caption \
  --keep_tokens=1 \
  --seed=42 \
  --max_data_loader_n_workers=1 \
  2>&1 | tee "{CONSOLE_LOG}"

**7.3. Exp01 — UNet-only training (bf16)**

Text encoder disabled to fit memory; resolution reduced to 768. Training succeeded but the
Turkish motif identity remained weak — motifs looked like generic decorative embroidery.

In [ ]:
#Eğitim verdiğim lora
from pathlib import Path
import os

BASE_DIR = Path("/content/drive/MyDrive/bitirme_lora")

TRAIN_DIR = BASE_DIR / "run_dataset"
OUTPUT_DIR = BASE_DIR / "output" / "turkmotiftesettur_sdxl_lora_v2_unet_bf16"
LOG_DIR = BASE_DIR / "logs" / "turkmotiftesettur_sdxl_lora_v2_unet_bf16"
CONSOLE_LOG = LOG_DIR / "console_train_log.txt"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

os.chdir("/content/sd-scripts")

!python sdxl_train_network.py \
  --pretrained_model_name_or_path="{base_model_path}" \
  --train_data_dir="{TRAIN_DIR}" \
  --output_dir="{OUTPUT_DIR}" \
  --logging_dir="{LOG_DIR}" \
  --output_name="turkmotiftesettur_sdxl_lora_v2_unet_bf16" \
  --save_model_as="safetensors" \
  --network_module=networks.lora \
  --network_train_unet_only \
  --network_dim=16 \
  --network_alpha=16 \
  --resolution=768,768 \
  --enable_bucket \
  --bucket_no_upscale \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --learning_rate=5e-5 \
  --unet_lr=5e-5 \
  --lr_scheduler=cosine \
  --optimizer_type=AdamW8bit \
  --max_train_epochs=4 \
  --save_every_n_epochs=1 \
  --mixed_precision=bf16 \
  --save_precision=fp16 \
  --xformers \
  --gradient_checkpointing \
  --cache_latents \
  --cache_latents_to_disk \
  --caption_extension=.txt \
  --shuffle_caption \
  --keep_tokens=1 \
  --max_grad_norm=1.0 \
  --seed=42 \
  --max_data_loader_n_workers=1 \
  2>&1 | tee "{CONSOLE_LOG}"

**7.4. Inspect Exp01 output weights (size + timestamp)**

In [ ]:
#Eğitim klasörlerinin ayarlanması

from pathlib import Path
import os
import time

output_dir = Path("/content/drive/MyDrive/bitirme_lora/output/turkmotiftesettur_sdxl_lora_v2_unet_bf16")

files = []
for f in output_dir.iterdir():
    if f.is_file():
        size_mb = os.path.getsize(f) / (1024 * 1024)
        mtime = os.path.getmtime(f)
        files.append((mtime, size_mb, f.name))

files = sorted(files, reverse=True)

for mtime, size_mb, name in files:
    print(time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(mtime)), f"{size_mb:.2f} MB", name)

**7.5. Archive Exp01 and record observation notes**

Weights, logs, and sample images are moved into a dedicated experiment folder, and a
`notes.txt` records what was learned — these observations shaped Exp02.

In [ ]:
#eğitim klasörlerinin düzenlenmesini sağlayan kod
from pathlib import Path
import shutil
import os

BASE_DIR = Path("/content/drive/MyDrive/bitirme_lora")

OLD_OUTPUT_DIR = BASE_DIR / "output" / "turkmotiftesettur_sdxl_lora_v2_unet_bf16"
OLD_LOG_DIR = BASE_DIR / "logs" / "turkmotiftesettur_sdxl_lora_v2_unet_bf16"

EXP1_DIR = BASE_DIR / "experiments" / "01_first_training_v2_unet_bf16"
EXP1_WEIGHTS = EXP1_DIR / "weights"
EXP1_LOGS = EXP1_DIR / "logs"
EXP1_SAMPLES = EXP1_DIR / "samples"

EXP1_WEIGHTS.mkdir(parents=True, exist_ok=True)
EXP1_LOGS.mkdir(parents=True, exist_ok=True)
EXP1_SAMPLES.mkdir(parents=True, exist_ok=True)

# Ağırlıkları kopyala
for f in OLD_OUTPUT_DIR.glob("*.safetensors"):
    shutil.copy2(f, EXP1_WEIGHTS / f.name)

# Logları kopyala
if OLD_LOG_DIR.exists():
    for f in OLD_LOG_DIR.glob("*"):
        if f.is_file():
            shutil.copy2(f, EXP1_LOGS / f.name)

# Test görsellerini de varsa kopyala
for f in BASE_DIR.glob("test_output*.png"):
    shutil.copy2(f, EXP1_SAMPLES / f.name)

notes = """
Experiment 01 - First Training

Model: SDXL LoRA
Training type: UNet-only
Precision: bf16
Resolution: 768x768
Network dim: 16
Observation:
- Tesettür / modest form learned.
- General luxury embroidery style learned.
- Turkish motif identity was not strong enough.
- Motifs appeared more like general floral/decorative embroidery.
"""

(EXP1_DIR / "notes.txt").write_text(notes.strip(), encoding="utf-8")

print("İlk eğitim güvenli şekilde arşivlendi:")
print(EXP1_DIR)

print("\nArşivlenen ağırlıklar:")
for f in EXP1_WEIGHTS.glob("*.safetensors"):
    print(f.name, round(os.path.getsize(f) / (1024 * 1024), 2), "MB")

**7.6. Link Exp02 dataset path**

In [ ]:
RUN_DATASET_DIR = BASE_DIR / "experiments" / "02_second_training_motif_stronger" / "dataset"

**7.7. Exp02 path setup (dataset on local disk for fast I/O)**

In [ ]:
from pathlib import Path
import os

BASE_DIR   = Path("/content/drive/MyDrive/bitirme_lora")
EXP02_DIR  = BASE_DIR / "experiments" / "02_second_training_motif_stronger"

TRAIN_DIR           = Path("/content/local_dataset")   # ← local
OUTPUT_DIR          = EXP02_DIR / "weights"
LOG_DIR             = EXP02_DIR / "logs"
CONSOLE_LOG         = LOG_DIR / "console_train_log.txt"
SAMPLE_PROMPTS_FILE = EXP02_DIR / "sample_prompts.txt"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

os.chdir("/content/sd-scripts")

train_dir           = str(TRAIN_DIR)
output_dir          = str(OUTPUT_DIR)
log_dir             = str(LOG_DIR)
console_log         = str(CONSOLE_LOG)
sample_prompts_file = str(SAMPLE_PROMPTS_FILE)

print("Eğitim şu dataset ile başlayacak:")
print(train_dir)

!python sdxl_train_network.py \
  --pretrained_model_name_or_path="{base_model_path}" \
  --train_data_dir="{train_dir}" \
  --output_dir="{output_dir}" \
  --logging_dir="{log_dir}" \
  --output_name="turkmotiftesettur_sdxl_lora_exp02_motif_stronger" \
  --save_model_as="safetensors" \
  --network_module=networks.lora \
  --network_dim=32 \
  --network_alpha=16 \
  --resolution=1024,1024 \
  --enable_bucket \
  --bucket_no_upscale \
  --min_bucket_reso=512 \
  --max_bucket_reso=1024 \
  --train_batch_size=2 \
  --gradient_accumulation_steps=2 \
  --learning_rate=1e-4 \
  --unet_lr=1e-4 \
  --text_encoder_lr=1e-5 \
  --lr_scheduler=cosine \
  --optimizer_type=AdamW8bit \
  --max_train_steps=1800 \
  --save_every_n_steps=300 \
  --mixed_precision=bf16 \
  --save_precision=fp16 \
  --xformers \
  --gradient_checkpointing \
  --cache_latents \
  --cache_latents_to_disk \
  --caption_extension=.txt \
  --shuffle_caption \
  --keep_tokens=1 \
  --max_grad_norm=1.0 \
  --seed=42 \
  --max_data_loader_n_workers=2 \
  --sample_prompts="{sample_prompts_file}" \
  --sample_every_n_steps=300 \
  2>&1 | tee "{console_log}"

**7.8. Exp02 training — final configuration**

Key improvements over Exp01: `network_dim` 16→32, text encoder re-enabled
(`text_encoder_lr=1e-5`), resolution back to 1024, step-based training (1800 steps),
gradient clipping (`max_grad_norm=1.0`), and dataset copied to the local Colab disk.
This version uses the TOML dataset config from Section 5.

In [ ]:
# Exp02 training using the TOML dataset config (same as the command-line version, but dataset settings come from the TOML file)

import os
os.chdir("/content/sd-scripts")

print("Dataset config ile eğitim başlayacak:")
print(DATASET_CONFIG)

!python sdxl_train_network.py \
  --pretrained_model_name_or_path="{base_model_path}" \
  --dataset_config="{DATASET_CONFIG}" \
  --output_dir="{OUTPUT_DIR}" \
  --logging_dir="{LOG_DIR}" \
  --output_name="turkmotiftesettur_sdxl_lora_exp02_motif_stronger" \
  --save_model_as="safetensors" \
  --network_module=networks.lora \
  --network_dim=32 \
  --network_alpha=16 \
  --learning_rate=1e-4 \
  --unet_lr=1e-4 \
  --text_encoder_lr=1e-5 \
  --lr_scheduler=cosine \
  --optimizer_type=AdamW8bit \
  --max_train_steps=1800 \
  --save_every_n_steps=300 \
  --mixed_precision=bf16 \
  --save_precision=fp16 \
  --xformers \
  --gradient_checkpointing \
  --cache_latents \
  --cache_latents_to_disk \
  --max_grad_norm=1.0 \
  --seed=42 \
  --max_data_loader_n_workers=2 \
  --sample_prompts="{SAMPLE_PROMPTS_FILE}" \
  --sample_every_n_steps=300 \
  2>&1 | tee "{CONSOLE_LOG}"

**7.9. (Optional) Inspect the training console log**

In [ ]:
#Eğitim çıktılarına baktığım
from pathlib import Path

log_file = Path("/content/drive/MyDrive/bitirme_lora/logs/turkmotiftesettur_sdxl_lora_v2_unet_bf16/console_train_log.txt")

if log_file.exists():
    lines = log_file.read_text(encoding="utf-8", errors="ignore").splitlines()
    print("\n".join(lines[-80:]))
else:
    print("Log dosyası bulunamadı.")

## 8. Inference — Image Generation

Load the trained Exp02 LoRA on top of the SDXL base model and generate designs. The LoRA
strength is controlled with `set_adapters` (0.9), and a fixed `torch.Generator` seed makes
results reproducible.

> If a `torchao` / version error occurs at load time, run the cleanup cell in Section 9.


**8.1. Generate an image with the Exp02 LoRA (single-file base model)**

In [ ]:
import torch
from diffusers import StableDiffusionXLPipeline
from pathlib import Path

base_model_path = "/content/drive/MyDrive/bitirme_lora/models/sd_xl_base_1.0.safetensors"

EXP2_DIR = Path("/content/drive/MyDrive/bitirme_lora/experiments/02_second_training_motif_stronger")
LORA_DIR = EXP2_DIR / "weights"
SAMPLE_DIR = EXP2_DIR / "samples"
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

lora_file = "turkmotiftesettur_sdxl_lora_exp02_motif_stronger.safetensors"

pipe = StableDiffusionXLPipeline.from_single_file(
    base_model_path,
    torch_dtype=torch.bfloat16,
    use_safetensors=True
).to("cuda")

pipe.load_lora_weights(
    str(LORA_DIR),
    weight_name=lora_file,
    adapter_name="exp02"
)

pipe.set_adapters(["exp02"], adapter_weights=[0.9])

prompt = (
    "turkmotiftesettur, elegant modern modest fashion, long modest dress, "
    "ottoman tulip motifs, iznik tile inspired embroidery, rumi and saz leaf patterns, "
    "blue and gold textile details, hijab, full body, fashion editorial, detailed fabric"
)

negative_prompt = (
    "low quality, blurry, bad hands, extra fingers, deformed, duplicate, "
    "text, watermark, ugly, distorted"
)

generator = torch.Generator(device="cuda").manual_seed(42)

image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=30,
    guidance_scale=7,
    width=1024,
    height=1024,
    generator=generator
).images[0]

save_path = SAMPLE_DIR / "test_exp02_final_seed42_weight09.png"
image.save(save_path)

print("Kaydedildi:", save_path)
image

**8.2. Alternative inference (downloads base model from Hugging Face)**

This variant uses `from_pretrained` so no local base-model file is required.

In [ ]:
import torch
from diffusers import StableDiffusionXLPipeline
from pathlib import Path

EXP2_DIR = Path("/content/drive/MyDrive/bitirme_lora/experiments/02_second_training_motif_stronger")
LORA_DIR = EXP2_DIR / "weights"
SAMPLE_DIR = EXP2_DIR / "samples"
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

lora_file = "turkmotiftesettur_sdxl_lora_exp02_motif_stronger.safetensors"

print("Base model yükleniyor (1-2 dk)...")
pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    use_safetensors=True
).to("cuda")

print("LoRA yükleniyor...")
pipe.load_lora_weights(
    str(LORA_DIR),
    weight_name=lora_file,
    adapter_name="exp02"
)

pipe.set_adapters(["exp02"], adapter_weights=[0.9])

prompt = (
    "turkmotiftesettur, single woman, one person, solo, luxurious modest kaftan gown, ottoman inspired gold and emerald embroidery, "
    "iznik tile floral patterns, rumi motif details, elegant draped fabric, sophisticated hijab, full body,"
    "high fashion editorial, dramatic studio lighting, highly detailed"
)

negative_prompt = (
    "multiple people, two women, two dresses, group, duplicate, twins, collage, "
    "split image, casual, plain, simple, low quality, blurry, bad hands, extra fingers, "
    "deformed, text, watermark, ugly, distorted, oversaturated, cartoon, plastic skin"
)

generator = torch.Generator(device="cuda").manual_seed(17)

print("Görsel üretiliyor...")
image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=30,
    guidance_scale=7,
    width=1024,
    height=1024,
    generator=generator
).images[0]

save_path = SAMPLE_DIR / "test_exp02_iznik_seed17.png"
image.save(save_path)

print("✅ Kaydedildi:", save_path)
image

## 9. Troubleshooting — Dependency Versions

Inference can fail due to version conflicts between `diffusers`, `transformers`, `peft`,
and `torchao`. The cells below resolve the issues encountered during the project:
removing the unused `torchao` package, and pinning a compatible set of versions that
loads Kohya LoRA weights (including the text-encoder part) correctly.


**9.1. Remove the incompatible/unused torchao package**

In [ ]:
!pip uninstall -y torchao

**9.2. Pin compatible library versions**

After running this, restart the runtime (Runtime → Restart session) before re-running
the inference cells.

In [ ]:
!pip install -q diffusers==0.27.2 peft==0.10.0 transformers==4.38.2 huggingface_hub==0.20.3 accelerate==0.27.2

**9.3. Check that the base model file exists on Drive**

In [ ]:
from pathlib import Path
#base modeli indirdim belbelki
base_model_path = Path("/content/drive/MyDrive/bitirme_lora/models/sd_xl_base_1.0.safetensors")

print("Base model var mı:", base_model_path.exists())
print(base_model_path)

## 10. Quantitative Evaluation

A systematic evaluation compares the base model, Exp01, and Exp02 across multiple prompts
and seeds using four complementary metric families:

| Metric | Measures | Better |
|--------|----------|--------|
| CLIP Score | text–image alignment (+ motif-reference alignment) | higher |
| DINOv2 Similarity | resemblance to the real dataset | higher |
| DINOv2 / LPIPS Diversity | output variety (no overfitting) | higher |
| FID / KID | distance between real and generated distributions | lower |

High diversity values are also the evidence that the model **generalizes** rather than
memorizing the training images.


**10.1. Set up evaluation paths and list available LoRA checkpoints**

In [ ]:
from pathlib import Path
import os, time
import pandas as pd

BASE_DIR = Path("/content/drive/MyDrive/bitirme_lora")

# Base model
BASE_MODEL_PATH = BASE_DIR / "models" / "sd_xl_base_1.0.safetensors"

# Exp01 için iki ihtimali de kontrol ediyoruz
EXP01_ARCHIVE = BASE_DIR / "experiments" / "01_first_training_v2_unet_bf16" / "weights"
EXP01_ORIGINAL = BASE_DIR / "output" / "turkmotiftesettur_sdxl_lora_v2_unet_bf16"

EXP01_WEIGHTS = EXP01_ARCHIVE if EXP01_ARCHIVE.exists() else EXP01_ORIGINAL

# Exp02
EXP02_DIR = BASE_DIR / "experiments" / "02_second_training_motif_stronger"
EXP02_WEIGHTS = EXP02_DIR / "weights"

# Metrik çıktıları
METRICS_DIR = BASE_DIR / "experiments" / "metrics_eval_all"
GEN_DIR = METRICS_DIR / "generated"
REPORT_DIR = METRICS_DIR / "reports"

GEN_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Base model:", BASE_MODEL_PATH, BASE_MODEL_PATH.exists())
print("Exp01 weights:", EXP01_WEIGHTS, EXP01_WEIGHTS.exists())
print("Exp02 weights:", EXP02_WEIGHTS, EXP02_WEIGHTS.exists())

def list_loras(folder):
    folder = Path(folder)
    files = sorted(folder.glob("*.safetensors"), key=lambda p: os.path.getmtime(p))
    return files

exp01_files = list_loras(EXP01_WEIGHTS)
exp02_files = list_loras(EXP02_WEIGHTS)

print("\nExp01 LoRA dosyaları:")
for f in exp01_files:
    print(time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(os.path.getmtime(f))), f.name)

print("\nExp02 LoRA dosyaları:")
for f in exp02_files:
    print(time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(os.path.getmtime(f))), f.name)

**10.2. Define the evaluation prompt set and seeds**

In [ ]:
eval_items = [
    {
        "id": "p01_ottoman_tulip",
        "base_prompt": "elegant modern modest fashion, long modest dress, ottoman tulip motifs, iznik tile inspired embroidery, rumi and saz leaf patterns, blue and gold textile details, hijab, full body, fashion editorial",
        "lora_prompt": "turkmotiftesettur, elegant modern modest fashion, long modest dress, ottoman tulip motifs, iznik tile inspired embroidery, rumi and saz leaf patterns, blue and gold textile details, hijab, full body, fashion editorial"
    },
    {
        "id": "p02_iznik_blue_white",
        "base_prompt": "modest evening dress, iznik tile inspired floral embroidery, white and blue patterns, hijab, full body, luxury fashion photography",
        "lora_prompt": "turkmotiftesettur, modest evening dress, iznik tile inspired floral embroidery, white and blue patterns, hijab, full body, luxury fashion photography"
    },
    {
        "id": "p03_anatolian_geometric",
        "base_prompt": "modern modest outfit, anatolian geometric motifs, embroidered long tunic and skirt, hijab, full body, studio fashion photo",
        "lora_prompt": "turkmotiftesettur, modern modest outfit, anatolian geometric motifs, embroidered long tunic and skirt, hijab, full body, studio fashion photo"
    },
    {
        "id": "p04_kaftan",
        "base_prompt": "kaftan style modest gown, ottoman inspired gold embroidery, elegant hijab, full body, luxury fashion lookbook",
        "lora_prompt": "turkmotiftesettur, kaftan style modest gown, ottoman inspired gold embroidery, elegant hijab, full body, luxury fashion lookbook"
    },
    {
        "id": "p05_daily_subtle",
        "base_prompt": "simple modern modest dress with subtle turkish embroidery, elegant daily outfit, hijab, full body, clean studio background",
        "lora_prompt": "turkmotiftesettur, simple modern modest dress with subtle turkish embroidery, elegant daily outfit, hijab, full body, clean studio background"
    }
]

seeds = [42, 101]

negative_prompt = "low quality, blurry, bad hands, extra fingers, deformed, duplicate, text, watermark, ugly, distorted"

**10.3. Build the model registry (base + all checkpoints)**

In [ ]:
def safe_model_name(name):
    return (
        name.replace(".safetensors", "")
            .replace("-", "_")
            .replace(".", "_")
            .replace(" ", "_")
    )

model_registry = []

model_registry.append({
    "model_group": "base",
    "model_name": "base_sdxl",
    "lora_dir": None,
    "lora_file": None,
    "adapter": None
})

for f in exp01_files:
    model_registry.append({
        "model_group": "exp01",
        "model_name": "exp01_" + safe_model_name(f.name),
        "lora_dir": str(f.parent),
        "lora_file": f.name,
        "adapter": "adapter_" + safe_model_name(f.name)
    })

for f in exp02_files:
    model_registry.append({
        "model_group": "exp02",
        "model_name": "exp02_" + safe_model_name(f.name),
        "lora_dir": str(f.parent),
        "lora_file": f.name,
        "adapter": "adapter_" + safe_model_name(f.name)
    })

pd.DataFrame(model_registry)

**10.4. Generate the full evaluation image grid**

For every model × prompt × seed combination, an image is generated and its metadata saved
to CSV. Already-generated images are skipped so the run can resume after interruptions.

In [ ]:
import torch
from diffusers import StableDiffusionXLPipeline
from pathlib import Path
import pandas as pd
import gc

device = "cuda" if torch.cuda.is_available() else "cpu"

pipe = StableDiffusionXLPipeline.from_single_file(
    str(BASE_MODEL_PATH),
    torch_dtype=torch.bfloat16,
    use_safetensors=True
).to(device)

pipe.set_progress_bar_config(disable=False)

metadata_rows = []

for model in model_registry:
    model_name = model["model_name"]
    model_out_dir = GEN_DIR / model_name
    model_out_dir.mkdir(parents=True, exist_ok=True)

    print("\n==============================")
    print("Üretiliyor:", model_name)

    # Önce varsa LoRA'yı kaldır
    try:
        pipe.unload_lora_weights()
    except Exception:
        pass

    if model["lora_file"] is not None:
        pipe.load_lora_weights(
            model["lora_dir"],
            weight_name=model["lora_file"],
            adapter_name=model["adapter"]
        )
        pipe.set_adapters([model["adapter"]], adapter_weights=[0.9])

    for item in eval_items:
        prompt = item["base_prompt"] if model["model_group"] == "base" else item["lora_prompt"]
        eval_prompt = item["base_prompt"]  # metrikte trigger kelimesini çıkarmak için bunu kullanacağız

        for seed in seeds:
            out_name = f"{item['id']}_seed{seed}.png"
            out_path = model_out_dir / out_name

            if out_path.exists():
                print("Var, atlandı:", out_path.name)
            else:
                generator = torch.Generator(device=device).manual_seed(seed)

                image = pipe(
                    prompt=prompt,
                    negative_prompt=negative_prompt,
                    num_inference_steps=30,
                    guidance_scale=7,
                    width=1024,
                    height=1024,
                    generator=generator
                ).images[0]

                image.save(out_path)
                print("Kaydedildi:", out_path.name)

            metadata_rows.append({
                "model_group": model["model_group"],
                "model_name": model_name,
                "lora_file": model["lora_file"],
                "prompt_id": item["id"],
                "seed": seed,
                "generation_prompt": prompt,
                "eval_prompt": eval_prompt,
                "image_path": str(out_path)
            })

    gc.collect()
    torch.cuda.empty_cache()

metadata_df = pd.DataFrame(metadata_rows)
metadata_path = REPORT_DIR / "generation_metadata.csv"
metadata_df.to_csv(metadata_path, index=False)

print("Metadata kaydedildi:", metadata_path)
metadata_df.head()
'''
metrics_eval_all/
├── generated/
│   ├── base_sdxl/
│   │   ├── p01_ottoman_tulip_seed42.png
│   │   ├── p01_ottoman_tulip_seed101.png
│   │   ├── p02_iznik_blue_white_seed42.png
│   │   └── ...
│   ├── exp01_turkmotiftesettur_..._000001/
│   │   ├── p01_ottoman_tulip_seed42.png
│   │   └── ...
│   ├── exp01_turkmotiftesettur_..._000002/
│   │   └── ...
│   ├── exp02_turkmotiftesettur_..._000300/
│   │   └── ...
│   └── ...
└── reports/
    └── generation_metadata.csv
'''

**10.5. CLIP Score — prompt alignment and motif-reference alignment**

In [ ]:
import torch
import pandas as pd
from PIL import Image
from pathlib import Path
from transformers import CLIPProcessor, CLIPModel
import torch.nn.functional as F

#metin ile görseli karşılaştırıp uyumlarına bakıyor.
device = "cuda" if torch.cuda.is_available() else "cpu"

clip_model_name = "openai/clip-vit-large-patch14"
clip_model = CLIPModel.from_pretrained(clip_model_name).to(device)
clip_processor = CLIPProcessor.from_pretrained(clip_model_name)

 #adil karşılaştırma için her görsel aynı promt ile karşılaştırıldı.
motif_text = (
    "traditional Turkish motif modest dress, Ottoman tulip motif, Iznik tile pattern, "
    "rumi motif, saz leaf motif, Anatolian embroidery, Turkish textile pattern, hijab fashion"
)
def clip_text_image_score(image_path, text):
    image = Image.open(image_path).convert("RGB")

    inputs = clip_processor(
        text=[text],
        images=image,
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = clip_model(**inputs)
        img_emb = F.normalize(outputs.image_embeds, dim=-1)
        txt_emb = F.normalize(outputs.text_embeds, dim=-1)
        return float((img_emb @ txt_emb.T).item())

clip_rows = []

for _, row in metadata_df.iterrows():
    image_path = row["image_path"]

    clip_score = clip_text_image_score(image_path, row["eval_prompt"])
    motif_clip_score = clip_text_image_score(image_path, motif_text)

    clip_rows.append({
        "model_group": row["model_group"],
        "model_name": row["model_name"],
        "lora_file": row["lora_file"],
        "prompt_id": row["prompt_id"],
        "seed": row["seed"],
        "clip_score": clip_score,
        "motif_clip_score": motif_clip_score
    })

clip_df = pd.DataFrame(clip_rows)
clip_path = REPORT_DIR / "clip_scores_per_image.csv"
clip_df.to_csv(clip_path, index=False)

print("CLIP skorları kaydedildi:", clip_path)
clip_df.head()
#her görseli hem kendi promtu ile hemde motif text promptu ile karşılaştırıp puanlıyoruz.

**10.6. DINOv2 — similarity to real data and output diversity**

In [ ]:
import torch
from PIL import Image
from pathlib import Path
import pandas as pd
import torch.nn.functional as F
from transformers import AutoImageProcessor, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"

dino_name = "facebook/dinov2-base"
dino_processor = AutoImageProcessor.from_pretrained(dino_name)
dino_model = AutoModel.from_pretrained(dino_name).to(device)
dino_model.eval()

REAL_CLOTHES_DIR = EXP2_DIR / "dataset" / "12_turkmotiftesettur_motif_clothes"
REAL_MOTIF_DIR = EXP2_DIR / "dataset" / "8_turkmotiftesettur_motif_references"

def get_image_paths(folder, limit=None):
    folder = Path(folder)
    paths = []
    for ext in ["*.jpg", "*.jpeg", "*.png", "*.webp"]:
        paths.extend(list(folder.glob(ext)))
    paths = sorted(paths)
    if limit:
        paths = paths[:limit]
    return paths

@torch.no_grad()
def dino_embed_images(paths, batch_size=8):
    embs = []

    for i in range(0, len(paths), batch_size):
        batch_paths = paths[i:i+batch_size]
        images = [Image.open(p).convert("RGB") for p in batch_paths]

        inputs = dino_processor(images=images, return_tensors="pt").to(device)
        outputs = dino_model(**inputs)

        # CLS token
        emb = outputs.last_hidden_state[:, 0, :]
        emb = F.normalize(emb, dim=-1)
        embs.append(emb.cpu())

    return torch.cat(embs, dim=0)

real_clothes_paths = get_image_paths(REAL_CLOTHES_DIR, limit=150)
real_motif_paths = get_image_paths(REAL_MOTIF_DIR, limit=150)

real_clothes_emb = dino_embed_images(real_clothes_paths)
real_motif_emb = dino_embed_images(real_motif_paths)

def dino_metrics_for_folder(folder):
    fake_paths = get_image_paths(folder)

    if len(fake_paths) == 0:
        return None

    fake_emb = dino_embed_images(fake_paths)

    # Her üretilen görselin gerçek kıyafet datasetindeki en yakın benzerliği
    sim_clothes = fake_emb @ real_clothes_emb.T
    max_sim_clothes = sim_clothes.max(dim=1).values.mean().item()

    # Her üretilen görselin motif referans datasetindeki en yakın benzerliği
    sim_motif = fake_emb @ real_motif_emb.T
    max_sim_motif = sim_motif.max(dim=1).values.mean().item()

    # Üretilen görseller kendi içinde ne kadar farklı?
    sim_self = fake_emb @ fake_emb.T
    n = sim_self.shape[0]
    mask = ~torch.eye(n, dtype=torch.bool)

    mean_self_similarity = sim_self[mask].mean().item()
    diversity = 1 - mean_self_similarity

    return {
        "dino_similarity_to_real_clothes_higher_better": max_sim_clothes,
        "dino_similarity_to_motif_refs_higher_better": max_sim_motif,
        "dino_diversity_higher_better": diversity
    }

dino_rows = []

for model in model_registry:
    model_name = model["model_name"]
    folder = GEN_DIR / model_name

    metrics = dino_metrics_for_folder(folder)
    if metrics is None:
        continue

    dino_rows.append({
        "model_group": model["model_group"],
        "model_name": model_name,
        "lora_file": model["lora_file"],
        **metrics
    })

dino_df = pd.DataFrame(dino_rows)
dino_path = REPORT_DIR / "dino_metrics.csv"
dino_df.to_csv(dino_path, index=False)

print("DINOv2 metrikleri kaydedildi:", dino_path)
dino_df.head()
#görseli gerçek görsel ilr, motif görselleri ile ve üretilen görselleri üretilen görsellerle kaştılatırıyor

**10.7. FID / KID — distribution distance**

In [ ]:
import torch
from pathlib import Path
from PIL import Image
from torchvision import transforms
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.kid import KernelInceptionDistance
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"

def load_metric_images(folder, max_images=200):
    folder = Path(folder)
    paths = []
    for ext in ["*.jpg", "*.jpeg", "*.png", "*.webp"]:
        paths.extend(list(folder.glob(ext)))
    paths = sorted(paths)[:max_images]

    if len(paths) == 0:
        raise ValueError(f"Görsel bulunamadı: {folder}")

    tfm = transforms.Compose([
        transforms.Resize((299, 299)),
        transforms.ToTensor()
    ])

    imgs = []
    for p in paths:
        img = Image.open(p).convert("RGB")
        x = tfm(img)
        x = (x * 255).byte()
        imgs.append(x)

    return torch.stack(imgs)

REAL_FOR_FID = REAL_CLOTHES_DIR

def calculate_fid_kid(real_dir, fake_dir):
    real = load_metric_images(real_dir, max_images=200)
    fake = load_metric_images(fake_dir, max_images=200)

    fid = FrechetInceptionDistance(feature=2048).to(device)
    fid.update(real.to(device), real=True)
    fid.update(fake.to(device), real=False)
    fid_score = fid.compute().item()

    subset_size = min(10, real.shape[0], fake.shape[0])

    kid = KernelInceptionDistance(
        subset_size=subset_size,
        subsets=20
    ).to(device)

    kid.update(real.to(device), real=True)
    kid.update(fake.to(device), real=False)
    kid_mean, kid_std = kid.compute()

    return fid_score, kid_mean.item(), kid_std.item()

fid_rows = []

for model in model_registry:
    model_name = model["model_name"]
    folder = GEN_DIR / model_name

    try:
        fid_score, kid_mean, kid_std = calculate_fid_kid(REAL_FOR_FID, folder)

        fid_rows.append({
            "model_group": model["model_group"],
            "model_name": model_name,
            "lora_file": model["lora_file"],
            "fid_lower_better": fid_score,
            "kid_mean_lower_better": kid_mean,
            "kid_std": kid_std
        })

        print(model_name, "FID:", fid_score, "KID:", kid_mean)

    except Exception as e:
        print("Atlandı:", model_name, e)

fid_df = pd.DataFrame(fid_rows)
fid_path = REPORT_DIR / "fid_kid_metrics.csv"
fid_df.to_csv(fid_path, index=False)

print("FID/KID kaydedildi:", fid_path)
fid_df.head()

**10.8. LPIPS — perceptual diversity**

In [ ]:
import lpips
import torch
from PIL import Image
from pathlib import Path
from torchvision import transforms
import pandas as pd
import itertools

device = "cuda" if torch.cuda.is_available() else "cpu"
#alexnet kullanarak insan gözüne yakın bir ölçüm yapmaya çalışır
lpips_model = lpips.LPIPS(net="alex").to(device)
lpips_model.eval()

lpips_tfm = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

def load_lpips_tensor(path):
    img = Image.open(path).convert("RGB")
    return lpips_tfm(img).unsqueeze(0).to(device)

def lpips_diversity(folder, max_pairs=60):
    paths = get_image_paths(folder)
    if len(paths) < 2:
        return None

    pairs = list(itertools.combinations(paths, 2))
    pairs = pairs[:max_pairs]

    scores = []
    with torch.no_grad():
        for a, b in pairs:
            xa = load_lpips_tensor(a)
            xb = load_lpips_tensor(b)
            score = lpips_model(xa, xb).item()
            scores.append(score)

    return sum(scores) / len(scores)

lpips_rows = []

for model in model_registry:
    model_name = model["model_name"]
    folder = GEN_DIR / model_name

    score = lpips_diversity(folder)

    if score is not None:
        lpips_rows.append({
            "model_group": model["model_group"],
            "model_name": model_name,
            "lora_file": model["lora_file"],
            "lpips_diversity_higher_better": score
        })

lpips_df = pd.DataFrame(lpips_rows)
lpips_path = REPORT_DIR / "lpips_diversity.csv"
lpips_df.to_csv(lpips_path, index=False)

print("LPIPS diversity kaydedildi:", lpips_path)
lpips_df.head()

## 11. Results Aggregation and Visualization

Load the per-metric CSV files, merge them, and produce summary tables and plots comparing
the base model, Exp01, and Exp02.


**11.1. Load metric CSV files**

In [ ]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path("/content/drive/MyDrive/bitirme_lora")
REPORT_DIR = BASE_DIR / "experiments" / "metrics_eval_all" / "reports"

clip_path = REPORT_DIR / "clip_scores_per_image.csv"
dino_path = REPORT_DIR / "dino_metrics.csv"
fid_path = REPORT_DIR / "fid_kid_metrics.csv"
lpips_path = REPORT_DIR / "lpips_diversity.csv"

print("CLIP var mı:", clip_path.exists())
print("DINO var mı:", dino_path.exists())
print("FID/KID var mı:", fid_path.exists())
print("LPIPS var mı:", lpips_path.exists())

dfs = {}

if clip_path.exists():
    clip_df = pd.read_csv(clip_path)
    clip_summary = clip_df.groupby(
        ["model_group", "model_name", "lora_file"],
        dropna=False
    ).agg(
        clip_score_mean=("clip_score", "mean"),
        motif_clip_score_mean=("motif_clip_score", "mean"),
        clip_score_std=("clip_score", "std"),
        motif_clip_score_std=("motif_clip_score", "std")
    ).reset_index()
    dfs["clip"] = clip_summary

if dino_path.exists():
    dfs["dino"] = pd.read_csv(dino_path)

if fid_path.exists():
    dfs["fid"] = pd.read_csv(fid_path)

if lpips_path.exists():
    dfs["lpips"] = pd.read_csv(lpips_path)

# İlk tabloyu başlat
if "clip" in dfs:
    final_df = dfs["clip"]
elif "fid" in dfs:
    final_df = dfs["fid"]
else:
    raise ValueError("Başlatmak için en az CLIP veya FID tablosu olmalı.")

# Diğerlerini birleştir
for key in ["dino", "fid", "lpips"]:
    if key in dfs and key != "clip":
        final_df = final_df.merge(
            dfs[key],
            on=["model_group", "model_name", "lora_file"],
            how="outer"
        )

rank_df = final_df.copy()

# Yüksek iyi olan metrikler
higher_better = [
    "clip_score_mean",
    "motif_clip_score_mean",
    "dino_similarity_to_real_clothes_higher_better",
    "dino_similarity_to_motif_refs_higher_better",
    "dino_diversity_higher_better",
    "lpips_diversity_higher_better"
]

# Düşük iyi olan metrikler
lower_better = [
    "fid_lower_better",
    "kid_mean_lower_better"
]

for col in higher_better:
    if col in rank_df.columns:
        rank_df[col + "_norm"] = (
            rank_df[col] - rank_df[col].min()
        ) / (
            rank_df[col].max() - rank_df[col].min() + 1e-9
        )

for col in lower_better:
    if col in rank_df.columns:
        rank_df[col + "_norm"] = 1 - (
            (rank_df[col] - rank_df[col].min()) /
            (rank_df[col].max() - rank_df[col].min() + 1e-9)
        )

score_cols = [c for c in rank_df.columns if c.endswith("_norm")]
rank_df["overall_score_higher_better"] = rank_df[score_cols].mean(axis=1)

rank_df = rank_df.sort_values("overall_score_higher_better", ascending=False)

final_report_path = REPORT_DIR / "ALL_MODEL_METRICS_REPORT.csv"
rank_df.to_csv(final_report_path, index=False)

print("Final rapor kaydedildi:", final_report_path)

display_cols = [
    "model_group",
    "model_name",
    "clip_score_mean",
    "motif_clip_score_mean",
    "dino_similarity_to_real_clothes_higher_better",
    "dino_similarity_to_motif_refs_higher_better",
    "fid_lower_better",
    "kid_mean_lower_better",
    "lpips_diversity_higher_better",
    "overall_score_higher_better"
]

display_cols = [c for c in display_cols if c in rank_df.columns]

rank_df[display_cols].head(20)

**11.2. Aggregate and visualize results (plots)**

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import textwrap

BASE_DIR = Path("/content/drive/MyDrive/bitirme_lora")
REPORT_DIR = BASE_DIR / "experiments" / "metrics_eval_all" / "reports"
PLOTS_DIR = REPORT_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

fid_path = REPORT_DIR / "fid_kid_metrics.csv"
lpips_path = REPORT_DIR / "lpips_diversity.csv"
clip_path = REPORT_DIR / "clip_scores_per_image.csv"
dino_path = REPORT_DIR / "dino_metrics.csv"
overall_path = REPORT_DIR / "ALL_MODEL_METRICS_REPORT.csv"

print("FID/KID:", fid_path.exists())
print("LPIPS:", lpips_path.exists())
print("CLIP:", clip_path.exists())
print("DINO:", dino_path.exists())
print("Overall:", overall_path.exists())


def shorten_name(name, max_len=38):
    name = str(name)
    name = name.replace("exp01_turkmotiftesettur_sdxl_lora_v2_unet_bf16", "exp01")
    name = name.replace("exp02_turkmotiftesettur_sdxl_lora_exp02_motif_stronger", "exp02")
    name = name.replace("_step", "_step")
    if len(name) > max_len:
        return name[:max_len] + "..."
    return name


def save_barh(df, x_col, y_col, title, xlabel, filename, ascending=True):
    plot_df = df.copy()
    plot_df = plot_df.sort_values(x_col, ascending=ascending)
    plot_df["short_name"] = plot_df[y_col].apply(shorten_name)

    plt.figure(figsize=(12, max(6, len(plot_df) * 0.45)))
    plt.barh(plot_df["short_name"], plot_df[x_col])
    plt.xlabel(xlabel)
    plt.title(title)
    plt.gca().invert_yaxis()

    for i, value in enumerate(plot_df[x_col]):
        plt.text(value, i, f" {value:.4f}", va="center", fontsize=8)

    plt.tight_layout()
    save_path = PLOTS_DIR / filename
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()

    print("Kaydedildi:", save_path)


# =========================
# 1) FID / KID grafikleri
# =========================

if fid_path.exists():
    fid_df = pd.read_csv(fid_path)

    save_barh(
        fid_df,
        x_col="fid_lower_better",
        y_col="model_name",
        title="FID Comparison - Lower is Better",
        xlabel="FID Score ↓",
        filename="01_fid_comparison.png",
        ascending=True
    )

    save_barh(
        fid_df,
        x_col="kid_mean_lower_better",
        y_col="model_name",
        title="KID Comparison - Lower is Better",
        xlabel="KID Mean ↓",
        filename="02_kid_comparison.png",
        ascending=True
    )


# =========================
# 2) LPIPS Diversity grafiği
# =========================

if lpips_path.exists():
    lpips_df = pd.read_csv(lpips_path)

    save_barh(
        lpips_df,
        x_col="lpips_diversity_higher_better",
        y_col="model_name",
        title="LPIPS Diversity Comparison - Higher is Better",
        xlabel="LPIPS Diversity ↑",
        filename="03_lpips_diversity.png",
        ascending=False
    )


# =========================
# 3) CLIPScore grafikleri
# =========================

if clip_path.exists():
    clip_df = pd.read_csv(clip_path)

    clip_summary = clip_df.groupby(
        ["model_group", "model_name", "lora_file"],
        dropna=False
    ).agg(
        clip_score_mean=("clip_score", "mean"),
        motif_clip_score_mean=("motif_clip_score", "mean")
    ).reset_index()

    save_barh(
        clip_summary,
        x_col="clip_score_mean",
        y_col="model_name",
        title="CLIPScore Comparison - Prompt/Image Alignment",
        xlabel="CLIPScore ↑",
        filename="04_clipscore_comparison.png",
        ascending=False
    )

    save_barh(
        clip_summary,
        x_col="motif_clip_score_mean",
        y_col="model_name",
        title="Motif CLIPScore Comparison - Turkish Motif Alignment",
        xlabel="Motif CLIPScore ↑",
        filename="05_motif_clipscore_comparison.png",
        ascending=False
    )


# =========================
# 4) DINOv2 grafikleri
# =========================

if dino_path.exists():
    dino_df = pd.read_csv(dino_path)

    if "dino_similarity_to_real_clothes_higher_better" in dino_df.columns:
        save_barh(
            dino_df,
            x_col="dino_similarity_to_real_clothes_higher_better",
            y_col="model_name",
            title="DINOv2 Similarity to Real Clothes Dataset - Higher is Better",
            xlabel="DINOv2 Similarity ↑",
            filename="06_dino_similarity_real_clothes.png",
            ascending=False
        )

    if "dino_similarity_to_motif_refs_higher_better" in dino_df.columns:
        save_barh(
            dino_df,
            x_col="dino_similarity_to_motif_refs_higher_better",
            y_col="model_name",
            title="DINOv2 Similarity to Motif References - Higher is Better",
            xlabel="DINOv2 Motif Similarity ↑",
            filename="07_dino_similarity_motif_refs.png",
            ascending=False
        )

    if "dino_diversity_higher_better" in dino_df.columns:
        save_barh(
            dino_df,
            x_col="dino_diversity_higher_better",
            y_col="model_name",
            title="DINOv2 Diversity - Higher is Better",
            xlabel="DINOv2 Diversity ↑",
            filename="08_dino_diversity.png",
            ascending=False
        )


# =========================
# 5) Overall Score grafiği
# =========================

if overall_path.exists():
    overall_df = pd.read_csv(overall_path)

    if "overall_score_higher_better" in overall_df.columns:
        save_barh(
            overall_df,
            x_col="overall_score_higher_better",
            y_col="model_name",
            title="Overall Model Ranking - Higher is Better",
            xlabel="Overall Score ↑",
            filename="09_overall_model_ranking.png",
            ascending=False
        )

print("\nTüm grafikler şu klasöre kaydedildi:")
print(PLOTS_DIR)

**11.3. Additional analysis / plots**

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import re
import numpy as np

BASE_DIR = Path("/content/drive/MyDrive/bitirme_lora")
REPORT_DIR = BASE_DIR / "experiments" / "metrics_eval_all" / "reports"
LINE_PLOTS_DIR = REPORT_DIR / "plots_line"
LINE_PLOTS_DIR.mkdir(parents=True, exist_ok=True)

fid_path = REPORT_DIR / "fid_kid_metrics.csv"
lpips_path = REPORT_DIR / "lpips_diversity.csv"
clip_path = REPORT_DIR / "clip_scores_per_image.csv"
dino_path = REPORT_DIR / "dino_metrics.csv"
overall_path = REPORT_DIR / "ALL_MODEL_METRICS_REPORT.csv"

print("FID/KID:", fid_path.exists())
print("LPIPS:", lpips_path.exists())
print("CLIP:", clip_path.exists())
print("DINO:", dino_path.exists())
print("Overall:", overall_path.exists())


def checkpoint_label(row):
    group = row["model_group"]
    name = str(row["model_name"])

    if group == "base":
        return "Base"

    if group == "exp01":
        if name.endswith("_000001"):
            return "Exp01-1"
        elif name.endswith("_000002"):
            return "Exp01-2"
        elif name.endswith("_000003"):
            return "Exp01-3"
        else:
            return "Exp01-Final"

    if group == "exp02":
        step_match = re.search(r"step0*([0-9]+)", name)
        if step_match:
            return f"Exp02-{int(step_match.group(1))}"
        else:
            return "Exp02-Final"

    return name


def checkpoint_order(row):
    group = row["model_group"]
    name = str(row["model_name"])

    if group == "base":
        return 0

    if group == "exp01":
        if name.endswith("_000001"):
            return 1
        elif name.endswith("_000002"):
            return 2
        elif name.endswith("_000003"):
            return 3
        else:
            return 4

    if group == "exp02":
        step_match = re.search(r"step0*([0-9]+)", name)
        if step_match:
            return int(step_match.group(1))
        else:
            return 1800

    return 999999


def prepare_metric_df(df):
    df = df.copy()
    df["checkpoint_label"] = df.apply(checkpoint_label, axis=1)
    df["checkpoint_order"] = df.apply(checkpoint_order, axis=1)
    return df


def plot_line_metric(df, metric_col, title, ylabel, filename, lower_better=False):
    plot_df = prepare_metric_df(df)

    plt.figure(figsize=(13, 6))

    # Base değerini ayrı gösterelim
    base_df = plot_df[plot_df["model_group"] == "base"]
    if len(base_df) > 0:
        base_value = base_df.iloc[0][metric_col]
        plt.axhline(
            y=base_value,
            linestyle="--",
            linewidth=1.5,
            label=f"Base SDXL = {base_value:.4f}"
        )

    for group in ["exp01", "exp02"]:
        gdf = plot_df[plot_df["model_group"] == group].sort_values("checkpoint_order")

        if len(gdf) == 0:
            continue

        x = list(range(1, len(gdf) + 1))
        y = gdf[metric_col].values
        labels = gdf["checkpoint_label"].values

        plt.plot(x, y, marker="o", linewidth=2, label=group.upper())

        for xi, yi, label in zip(x, y, labels):
            plt.text(xi, yi, f"{label}\n{yi:.4f}", fontsize=8, ha="center", va="bottom")

    plt.title(title)
    plt.xlabel("Checkpoint Progression")
    plt.ylabel(ylabel)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()

    save_path = LINE_PLOTS_DIR / filename
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()

    print("Kaydedildi:", save_path)


# =========================
# FID / KID çizgi grafikleri
# =========================

if fid_path.exists():
    fid_df = pd.read_csv(fid_path)

    plot_line_metric(
        fid_df,
        metric_col="fid_lower_better",
        title="FID Trend Across Checkpoints - Lower is Better",
        ylabel="FID Score ↓",
        filename="01_fid_line_trend.png",
        lower_better=True
    )

    plot_line_metric(
        fid_df,
        metric_col="kid_mean_lower_better",
        title="KID Trend Across Checkpoints - Lower is Better",
        ylabel="KID Mean ↓",
        filename="02_kid_line_trend.png",
        lower_better=True
    )


# =========================
# LPIPS Diversity çizgi grafiği
# =========================

if lpips_path.exists():
    lpips_df = pd.read_csv(lpips_path)

    plot_line_metric(
        lpips_df,
        metric_col="lpips_diversity_higher_better",
        title="LPIPS Diversity Trend Across Checkpoints - Higher is Better",
        ylabel="LPIPS Diversity ↑",
        filename="03_lpips_diversity_line_trend.png"
    )


# =========================
# CLIPScore çizgi grafikleri
# =========================

if clip_path.exists():
    clip_df = pd.read_csv(clip_path)

    clip_summary = clip_df.groupby(
        ["model_group", "model_name", "lora_file"],
        dropna=False
    ).agg(
        clip_score_mean=("clip_score", "mean"),
        motif_clip_score_mean=("motif_clip_score", "mean")
    ).reset_index()

    plot_line_metric(
        clip_summary,
        metric_col="clip_score_mean",
        title="CLIPScore Trend Across Checkpoints - Higher is Better",
        ylabel="CLIPScore ↑",
        filename="04_clipscore_line_trend.png"
    )

    plot_line_metric(
        clip_summary,
        metric_col="motif_clip_score_mean",
        title="Motif CLIPScore Trend Across Checkpoints - Higher is Better",
        ylabel="Motif CLIPScore ↑",
        filename="05_motif_clipscore_line_trend.png"
    )


# =========================
# DINOv2 çizgi grafikleri
# =========================

if dino_path.exists():
    dino_df = pd.read_csv(dino_path)

    if "dino_similarity_to_real_clothes_higher_better" in dino_df.columns:
        plot_line_metric(
            dino_df,
            metric_col="dino_similarity_to_real_clothes_higher_better",
            title="DINOv2 Similarity to Real Clothes Trend - Higher is Better",
            ylabel="DINO Similarity to Real Clothes ↑",
            filename="06_dino_real_clothes_line_trend.png"
        )

    if "dino_similarity_to_motif_refs_higher_better" in dino_df.columns:
        plot_line_metric(
            dino_df,
            metric_col="dino_similarity_to_motif_refs_higher_better",
            title="DINOv2 Similarity to Motif References Trend - Higher is Better",
            ylabel="DINO Similarity to Motif References ↑",
            filename="07_dino_motif_refs_line_trend.png"
        )

    if "dino_diversity_higher_better" in dino_df.columns:
        plot_line_metric(
            dino_df,
            metric_col="dino_diversity_higher_better",
            title="DINOv2 Diversity Trend Across Checkpoints - Higher is Better",
            ylabel="DINO Diversity ↑",
            filename="08_dino_diversity_line_trend.png"
        )


# =========================
# Overall Score çizgi grafiği
# =========================

if overall_path.exists():
    overall_df = pd.read_csv(overall_path)

    if "overall_score_higher_better" in overall_df.columns:
        plot_line_metric(
            overall_df,
            metric_col="overall_score_higher_better",
            title="Overall Model Success Trend - Higher is Better",
            ylabel="Overall Score ↑",
            filename="09_overall_score_line_trend.png"
        )

print("\nÇizgi grafikler kaydedildi:")
print(LINE_PLOTS_DIR)

**11.4. Final comparison plots**

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import re

BASE_DIR = Path("/content/drive/MyDrive/bitirme_lora")
REPORT_DIR = BASE_DIR / "experiments" / "metrics_eval_all" / "reports"
LINE_PLOTS_DIR = REPORT_DIR / "plots_line"
LINE_PLOTS_DIR.mkdir(parents=True, exist_ok=True)

fid_df = pd.read_csv(REPORT_DIR / "fid_kid_metrics.csv")

exp02_df = fid_df[fid_df["model_group"] == "exp02"].copy()

def extract_step(name):
    name = str(name)
    match = re.search(r"step0*([0-9]+)", name)
    if match:
        return int(match.group(1))
    return 1800

exp02_df["step"] = exp02_df["model_name"].apply(extract_step)
exp02_df = exp02_df.sort_values("step")

plt.figure(figsize=(12, 6))

plt.plot(exp02_df["step"], exp02_df["fid_lower_better"], marker="o", linewidth=2, label="FID ↓")
plt.plot(exp02_df["step"], exp02_df["kid_mean_lower_better"] * 3000, marker="o", linewidth=2, label="KID × 3000 ↓")

for _, row in exp02_df.iterrows():
    plt.text(row["step"], row["fid_lower_better"], f'{row["fid_lower_better"]:.1f}', fontsize=8)

plt.title("Exp02 Training Progress - FID and KID Trend")
plt.xlabel("Training Step")
plt.ylabel("Metric Value")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

save_path = LINE_PLOTS_DIR / "10_exp02_fid_kid_training_progress.png"
plt.savefig(save_path, dpi=200, bbox_inches="tight")
plt.show()

print("Kaydedildi:", save_path)